<a href="https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
from pathlib import Path
import numpy as np
import pandas as pd

CANDIDATES = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
]
CSV_PATH = next((p for p in CANDIDATES if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "Can't find content_refresh_anonymized.csv. Run this notebook from the repo "
        "(work/notebooks/) so the ../../data/raw/ path resolves."
    )

raw = pd.read_csv(CSV_PATH)
df = raw.copy()
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"rows: {len(df):,}   clients: {df['client_id'].nunique()}")

key_fields = ["impressions_90d", "clicks_90d", "sessions_90d", "word_count", "ctr", "avg_position", "days_since_last_update"]
dist = df[key_fields].describe(percentiles=[0.5, 0.9, 0.99]).T[["min", "50%", "90%", "99%", "max"]]
dist.columns = ["min", "p50", "p90", "p99", "max"]
print("\ndistribution of key fields (note how far p99 sits from p50 -- heavy tails):")
print(dist.round(2))

print("\nheavy-tail check: mean vs median (a mean far above the median = a few giants pulling it up)")
for c in ["impressions_90d", "clicks_90d", "sessions_90d"]:
    print(f"  {c}: mean={df[c].mean():.1f}  median={df[c].median():.1f}  ratio={df[c].mean()/max(df[c].median(),1):.1f}x")

print("\nzero/blank flags worth knowing before testing anything:")
print(f"  avg_position == 0 (means 'no position data', not position zero): {(df['avg_position']==0).sum():,} rows")
print(f"  clicks_90d == 0: {(df['clicks_90d']==0).sum():,} rows")
print(f"  word_count blank: {df['word_count'].isna().sum():,} rows")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


rows: 30,000   clients: 32

distribution of key fields (note how far p99 sits from p50 -- heavy tails):
                        min      p50       p90       p99       max
impressions_90d         1.0   731.00  12136.40  73505.83  517715.0
clicks_90d              0.0     1.00     32.00    253.01    4178.0
sessions_90d            1.0     7.00     88.00    451.01    4345.0
word_count              8.0  2877.00   5327.00   7292.00    9546.0
ctr                     0.0     0.07      0.65      8.33     100.0
avg_position            0.0    10.80     36.80     69.90     245.0
days_since_last_update  1.0    20.00    104.00    106.00     373.0

heavy-tail check: mean vs median (a mean far above the median = a few giants pulling it up)
  impressions_90d: mean=5200.4  median=731.0  ratio=7.1x
  clicks_90d: mean=16.1  median=1.0  ratio=16.1x
  sessions_90d: mean=37.1  median=7.0  ratio=5.3x

zero/blank flags worth knowing before testing anything:
  avg_position == 0 (means 'no position data', not pos

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# ===== Signal test #1: "Longer content gets more organic visibility" =====
t1 = df.dropna(subset=["word_count_tier"]).groupby("word_count_tier").agg(
    n=("content_id", "size"), median_impressions_90d=("impressions_90d", "median"),
).reindex(["<1000", "1000-2000", "2000-3500", "3500+"])
print("Signal #1 -- 'longer content gets more organic visibility'")
print(t1)
verdict1 = "CONFIRMED"
print(f"n's all well above the 50-row floor. Median impressions rises monotonically with word count "
      f"tier ({t1['median_impressions_90d'].iloc[0]:.0f} -> {t1['median_impressions_90d'].iloc[-1]:.0f}). "
      f"Verdict: {verdict1} -- directional, not causal (longer pages may also be older/better-optimized).\n")

# ===== Signal test #2: "Higher search-volume keywords carry a higher CPC" =====
sv = df.dropna(subset=["search_volume", "cpc"])
sv = sv[sv["search_volume"] > 0].copy()
sv["sv_bucket"] = pd.qcut(sv["search_volume"], 4, duplicates="drop")
t2 = sv.groupby("sv_bucket").agg(n=("cpc", "size"), median_cpc=("cpc", "median"))
spearman = sv["search_volume"].corr(sv["cpc"], method="spearman")
print("Signal #2 -- 'higher search-volume keywords carry a higher CPC'")
print(t2)
print(f"Spearman rank correlation (search_volume, cpc): {spearman:.3f}, n={len(sv):,}")
verdict2 = "MIXED"
print(f"Verdict: {verdict2} -- positive and n's are large, but the increase is concentrated almost "
      f"entirely in the top quartile (median CPC ~0 for the bottom two buckets); not a clean linear story.\n")

# ===== Signal test #3: "Transactional/commercial intent gets a higher CTR than informational" =====
# weighted CTR = sum(clicks) / sum(impressions), NOT the mean of each page's own CTR
mi = df.dropna(subset=["main_intent"])
t3 = mi.groupby("main_intent").apply(
    lambda g: pd.Series({"n": len(g), "weighted_ctr_pct": g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100})
)
print("Signal #3 -- 'transactional/commercial intent gets higher CTR than informational'")
print(t3.round(3))
verdict3 = "CONFIRMED (informational vs transactional/commercial); navigational excluded"
print(f"navigational has n={int(t3.loc['navigational','n'])} -- below the ~50-row floor, so no verdict there. "
      f"transactional (0.36%) and commercial (0.31%) both beat informational (0.29%), consistent with intent theory. "
      f"Verdict: {verdict3}.")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Signal #1 -- 'longer content gets more organic visibility'
                     n  median_impressions_90d
word_count_tier                               
<1000              973                     4.0
1000-2000         3780                   172.0
2000-3500        11263                   997.0
3500+             6285                  1340.0
n's all well above the 50-row floor. Median impressions rises monotonically with word count tier (4 -> 1340). Verdict: CONFIRMED -- directional, not causal (longer pages may also be older/better-optimized).

Signal #2 -- 'higher search-volume keywords carry a higher CPC'
                    n  median_cpc
sv_bucket                        
(9.999, 20.0]    9601        0.00
(20.0, 70.0]     3339        0.01
(70.0, 74000.0]  3511        0.45
Spearman rank correlation (search_volume, cpc): 0.424, n=16,451
Verdict: MIXED -- positive and n's are large, but the increase is concentrated almost entirely in the top quartile (median CPC ~0 for the bottom two buck

/tmp/ipykernel_2737/563769526.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  t2 = sv.groupby("sv_bucket").agg(n=("cpc", "size"), median_cpc=("cpc", "median"))
/tmp/ipykernel_2737/563769526.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  t3 = mi.groupby("main_intent").apply(


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# The flag: scripts/02_baseline_score.py's reason_codes() sets "stale_visible_page" when
#   days_since_last_update >= 180  AND  impressions_90d >= 500
# Assumption baked into that rule: staleness (time since last update) predicts decline.
# Test it at two grain levels: the flag's own exact definition, and the broader freshness_tier.

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# 1) The flag's own exact cell
visible = df[df["impressions_90d"] >= 500].copy()
visible["is_stale_flag"] = visible["days_since_last_update"] >= 180
flag_cell = visible.groupby("is_stale_flag").agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
print("Flag's exact definition (impressions_90d >= 500, split by days_since_last_update >= 180):")
print(flag_cell.round(3))

n_stale_flag = int(flag_cell.loc[True, "n"]) if True in flag_cell.index else 0
print(f"\nThe flag's own 'stale' cell has n={n_stale_flag} rows -- below the ~50-row floor from the skill. "
      f"No verdict can honestly be drawn on the flag's exact rule from this slice.")

# 2) Broader freshness_tier, no visibility filter, to see if the underlying belief holds at all
t = df.groupby("freshness_tier").agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean")).reindex(
    ["0-30", "31-90", "91-180", "181+"]
)
print("\nBroader check -- freshness_tier vs decline rate, full population (bigger n's):")
print(t.round(3))

print(
    f"\nIf staleness->decline were a clean linear story, decline_rate should rise 0-30 -> 31-90 -> 91-180 -> 181+. "
    f"Instead it PEAKS at 91-180 ({t.loc['91-180','decline_rate']:.3f}) and comes back DOWN at 181+ "
    f"({t.loc['181+','decline_rate']:.3f}), which is even lower than the freshest bucket 0-30 "
    f"({t.loc['0-30','decline_rate']:.3f})."
)

verdict3 = "MIXED / INSUFFICIENT DATA"
print(f"\nVerdict: {verdict3} -- the flag's own precise cell (n={n_stale_flag}) is too small to test directly in "
      f"this slice; and the broader relationship isn't the monotonic 'older = worse' story the flag assumes -- "
      f"it's non-monotonic, peaking mid-range. The rule may still be a reasonable heuristic (it never fires wrong "
      f"more than it fires right, per the base decline rate), but the data doesn't confirm its assumption cleanly.")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Flag's exact definition (impressions_90d >= 500, split by days_since_last_update >= 180):
                   n  decline_rate
is_stale_flag                     
False          16709         0.595
True              17         0.941

The flag's own 'stale' cell has n=17 rows -- below the ~50-row floor from the skill. No verdict can honestly be drawn on the flag's exact rule from this slice.

Broader check -- freshness_tier vs decline rate, full population (bigger n's):
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471

If staleness->decline were a clean linear story, decline_rate should rise 0-30 -> 31-90 -> 91-180 -> 181+. Instead it PEAKS at 91-180 (0.611) and comes back DOWN at 181+ (0.471), which is even lower than the freshest bucket 0-30 (0.511).

Verdict: MIXED / INSUFFICIENT DATA -- the flag's own precise cell (n=17) 

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Nothing new to compute -- this cell just states, in numbers, the evidence behind the
# "what this means in practice" note above, so the claim isn't just prose.
print("Evidence recap for the practice note:")
print(f"  - Signal #1 (word count -> visibility): CONFIRMED, n={len(df.dropna(subset=['word_count_tier'])):,}")
print(f"  - Signal #2 (search volume -> CPC): MIXED, n={len(sv):,}, spearman={spearman:.3f}")
print(f"  - Signal #3 (intent -> CTR): CONFIRMED for commercial/transactional vs informational")
print(f"  - Flag test (staleness -> decline): MIXED/INSUFFICIENT DATA, flag's own cell n={n_stale_flag}")
print(
    "\nTakeaway: word count and search-intent signals are trustworthy enough to lean on for "
    "prioritization; the 'stale + visible' refresh flag is not yet backed by enough same-slice "
    "evidence to treat as a strong rule -- more data or a lower impression threshold is needed "
    "before a content team should act on it with confidence."
)


# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Evidence recap for the practice note:
  - Signal #1 (word count -> visibility): CONFIRMED, n=22,301
  - Signal #2 (search volume -> CPC): MIXED, n=16,451, spearman=0.424
  - Signal #3 (intent -> CTR): CONFIRMED for commercial/transactional vs informational
  - Flag test (staleness -> decline): MIXED/INSUFFICIENT DATA, flag's own cell n=17

Takeaway: word count and search-intent signals are trustworthy enough to lean on for prioritization; the 'stale + visible' refresh flag is not yet backed by enough same-slice evidence to treat as a strong rule -- more data or a lower impression threshold is needed before a content team should act on it with confidence.


## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.